# Species Prediction with Evo 2 7B — Pipeline README

**Project:** What Do DNA Language Models Learn? (Species prediction task)  
**Model:** Evo 2 7B (`arcinstitute/evo-2-7b`)  
**GPU budget:** 46 GB (single GPU, shared server)

---

## Data you need to download

| File | Source | Where it goes |
|---|---|---|
| `bac120_metadata_r232.tsv.gz` | https://data.gtdb.ecogenomic.org/releases/release232/232.0/ | `data/gtdb/` |
| `ar53_metadata_r232.tsv.gz` | same | `data/gtdb/` |
| `bac120_taxonomy_r232.tsv.gz` | same | `data/gtdb/` |
| `ar53_taxonomy_r232.tsv.gz` | same | `data/gtdb/` |
| `bac120.tree.gz` | same | `data/trees/` |
| `ar53.tree.gz` | same | `data/trees/` |
| NCBI Datasets CLI | https://www.ncbi.nlm.nih.gov/datasets/docs/v2/download-and-install/ | anywhere on PATH |
| Per-genome FASTAs | downloaded automatically by `01_subsample_species.py` | `data/ncbi_genomes/` |

The GTDB metadata files are ~500 MB total. The tree files are ~200 MB.  
Individual genome FASTAs average 2–4 MB each (bacterial).  
For 200 genomes (20 phyla × 10 species), expect ~1 GB of FASTAs.

---

## Pipeline overview

```
00_download_data.sh          # Download GTDB files and NCBI Datasets CLI
01_subsample_species.py      # Pick representative species + download FASTAs
02_extract_fragments.py      # Extract 1 kb windows from each genome
03_build_phylo_distance.py   # Compute pairwise phylogenetic distance matrix
04_extract_embeddings_evo2.py  # Forward-pass Evo 2 7B, cache layer embeddings
05_train_probes.py           # Train probes + evaluate both tasks
06_umap_visualization.py     # UMAP plots colored by phylum/domain
```

---

## Step-by-step run instructions

### 0. Environment setup

```bash
conda create -n evo2probe python=3.11 -y
conda activate evo2probe
pip install -r requirements.txt
```

### 1. Download GTDB metadata and trees

```bash
bash 00_download_data.sh
```

### 2. Subsample species and download genomes

```bash
# Default: 20 phyla × 10 species = 200 genomes (~1 GB)
python 01_subsample_species.py \
    --gtdb_dir data/gtdb \
    --out_dir data \
    --n_phyla 20 \
    --species_per_phylum 10 \
    --datasets_bin /path/to/datasets   # NCBI Datasets CLI
```

> **Tip for shared servers:** add `--skip_download` and download the FASTAs  
> overnight via a separate `nohup` job using `accessions_to_download.txt`.

### 3. Extract fragments

```bash
python 02_extract_fragments.py \
    --species_index data/species_index.tsv \
    --genome_dir data/ncbi_genomes \
    --out_dir data/fragments \
    --frag_len 1024 \
    --frags_per_genome 50
```

This produces `data/fragments/fragments.tsv` (~10k rows for 200 genomes).

### 4. Build phylogenetic distance matrix

```bash
python 03_build_phylo_distance.py \
    --tree_dir data/trees \
    --species_index data/species_index.tsv \
    --out_dir data/phylo
```

For 200 species this takes ~1 minute. The output is a compressed `.npz` file.

### 5. Extract Evo 2 embeddings (GPU-intensive step)

```bash
python 04_extract_embeddings_evo2.py \
    --fragments data/fragments/fragments.tsv \
    --out_dir data/embeddings/evo2 \
    --batch_size 1 \
    --frag_len 1024 \
    --device cuda
```

**Memory notes:**  
- Evo 2 7B in fp16 = ~14 GB weights  
- Each forward pass at 1024 bp ≈ 4–6 GB activations  
- Total peak ≈ 20–22 GB → well within 46 GB  
- If you see OOMs, ensure no other jobs are on the GPU (`nvidia-smi`)  
- The script saves a `.cursor` file, so `--resume` restarts from the last batch  

For 10,000 fragments at batch_size=1 on an A100, expect ~3–5 hours.  
Use a SLURM job:

```bash
srun --gpus=1 --mem=48G --time=08:00:00 \
    python 04_extract_embeddings_evo2.py --batch_size 1
```

### 6. Train probes and evaluate

```bash
python 05_train_probes.py \
    --embeddings_dir data/embeddings/evo2 \
    --fragments data/fragments/fragments.tsv \
    --phylo_dir data/phylo \
    --out_dir results
```

By default runs both logistic regression and MLP probes at every layer.  
Add `--skip_mlp` for a fast first pass with logistic only.  
Add `--probe_layers 0,12,24` to probe only selected layers.

**Key output files:**
- `results/classification_results.tsv` — F1/AUC/AUPRC per (layer, probe)
- `results/phylo_correlation.tsv` — Spearman ρ per layer
- `results/layer_curves.png` — classification performance across layers
- `results/phylo_curves.png` — phylogenetic correlation across layers

### 7. UMAP visualization

```bash
# Visualize embedding layer (0), middle layer, and final layer
N_LAYERS=$(ls data/embeddings/evo2/layer_*.h5 | wc -l)
MID=$(( N_LAYERS / 2 ))
LAST=$(( N_LAYERS - 1 ))

python 06_umap_visualization.py \
    --embeddings_dir data/embeddings/evo2 \
    --fragments data/fragments/fragments.tsv \
    --out_dir results/umap \
    --layers "0,${MID},${LAST}"
```

---

## Memory-saving options

| Situation | Fix |
|---|---|
| OOM during embedding extraction | `--batch_size 1` (already default); reduce `--frag_len` to 512 |
| Too many fragments to fit in RAM during probing | `--probe_layers 0,12,24` to probe only a subset |
| Slow UMAP | `--max_points 2000` |
| Phylo tree takes too long | Use only bacterial tree (`bac120.tree`) initially |

---

## Expected results (targets from proposal)

| Metric | Feasible target | Ambitious target |
|---|---|---|
| Species classification F1 (best layer) | > baseline k-mer TF-IDF | AUPRC substantial improvement |
| Phylo Spearman ρ (cosine, best layer) | ρ > 0.3 | ρ > 0.6 |

---

## File structure after full run

```
data/
  gtdb/               ← GTDB metadata TSVs
  trees/              ← GTDB Newick trees
  ncbi_genomes/       ← Per-accession FASTA directories
  fragments/
    fragments.tsv     ← All extracted fragments
    species_split.json
  phylo/
    distance_matrix.npz
  embeddings/
    evo2/
      layer_00.h5 … layer_NN.h5
      frag_ids.txt
results/
  classification_results.tsv
  phylo_correlation.tsv
  layer_curves.png
  phylo_curves.png
  umap/
    layer_00_phylum.png  …
```
